Pull data from SSMS

In [85]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

server = "localhost"
database = "RevenueAnalytics"

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

OUTPUT_DIR_FEATURE_ENGINEERING = Path("../data/feature engineering")
OUTPUT_DIR_PREPROCESSED = Path("../data/preprocessed")
OUTPUT_DIR_FEATURE_ENGINEERING.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_PREPROCESSED.mkdir(parents=True, exist_ok=True)
print(f"Output folder preprocessed: {OUTPUT_DIR_PREPROCESSED.resolve()}")
print(f"Output folder feature engineering: {OUTPUT_DIR_FEATURE_ENGINEERING.resolve()}")

Output folder preprocessed: C:\Users\Anast\OneDrive\Desktop\AS Portfolio\early-warning\data\preprocessed
Output folder feature engineering: C:\Users\Anast\OneDrive\Desktop\AS Portfolio\early-warning\data\feature engineering


In [ ]:
# connection ssms
connection_url = URL.create(
    "mssql+pyodbc",
    host=server,
    database=database,
    query={
        "driver": "ODBC Driver 18 for SQL Server",
        "trusted_connection": "yes",
        "TrustServerCertificate": "yes"})

engine = create_engine(connection_url)

# verify connection
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT @@SERVERNAME AS server_name, DB_NAME() AS database_name"))
    
    print(result.fetchone())

C:\Users\Anast\AppData\Local\Temp\ipykernel_10936\674463289.py:2: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with engine.connect() as connection:


('Ana', 'RevenueAnalytics')


In [ ]:
# load first rows from sales table
query = """
SELECT TOP 100 *
FROM raw.factSales
"""

with engine.connect() as connection:
    df = pd.read_sql(text(query), connection)

df.head()

,sales_id,order_id,date_id,date,year_month,customer_id,product_id,region_id,sales_rep_id,units,asp_eur,discount_pct,revenue_eur,revenue_local_currency,currency,unit_cost_eur,gross_profit_eur,gross_margin_pct,is_outlier_order
0,1,ORD-00000001,20220304,2022-03-04,2022-03-01 00:00:00.0000000,282,1133,2,39,26,104.00,0.092,2704.13,2704.13,EUR,78.47,663.79,0.2455,False
1,2,ORD-00000002,20240503,2024-05-03,2024-05-01 00:00:00.0000000,562,1105,2,73,21,1698.07,0.089,35659.45,35659.45,EUR,944.57,15823.42,0.4437,False
2,3,ORD-00000003,20241205,2024-12-05,2024-12-01 00:00:00.0000000,146,1007,7,41,117,983.15,0.123,115029.00,169160.29,CAD,637.92,40392.64,0.3512,False
3,4,ORD-00000004,20241119,2024-11-19,2024-11-01 00:00:00.0000000,487,1176,6,10,5,63.48,0.084,317.39,344.99,USD,55.67,39.04,0.1230,False
4,5,ORD-00000005,20220713,2022-07-13,2022-07-01 00:00:00.0000000,869,1104,10,81,61,931.87,0.157,56844.25,9168427.42,JPY,728.64,12397.02,0.2181,False


In [13]:
# load data in df
sales_transactions = pd.read_sql_table(
    table_name="factSales",
    schema="staging",
    con=engine)

sales_transactions.head()

,sales_id,order_id,date_id,order_date,year_month,customer_id,product_id,region_id,sales_rep_id,units,...,gross_profit_eur,gross_margin_pct,is_outlier_order,cogs_eur,discount_value_eur,margin_category,order_size_category,missing_sales_rep_flag,missing_discount_flag,missing_margin_flag
0,1,ORD-00000001,20220304,2022-03-04,2022-03-01,282,1133,2,39.0,26,...,663.79,0.2455,False,2040.34,248.77996,Low Margin,Medium Order,0,0,0
1,2,ORD-00000002,20240503,2024-05-03,2024-05-01,562,1105,2,73.0,21,...,15823.42,0.4437,False,19836.03,3173.69105,High Margin,Small Order,0,0,0
2,3,ORD-00000003,20241205,2024-12-05,2024-12-01,146,1007,7,41.0,117,...,40392.64,0.3512,False,74636.36,14148.56700,Medium Margin,Large Order,0,0,0
3,4,ORD-00000004,20241119,2024-11-19,2024-11-01,487,1176,6,10.0,5,...,39.04,0.1230,False,278.35,26.66076,Low Margin,Small Order,0,0,0
4,5,ORD-00000005,20220713,2022-07-13,2022-07-01,869,1104,10,81.0,61,...,12397.02,0.2181,False,44447.23,8924.54725,Low Margin,Medium Order,0,0,0


In [14]:
sales_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 26 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   sales_id                120000 non-null  int64         
 1   order_id                120000 non-null  object        
 2   date_id                 120000 non-null  int64         
 3   order_date              120000 non-null  datetime64[ns]
 4   year_month              120000 non-null  datetime64[ns]
 5   customer_id             120000 non-null  int64         
 6   product_id              120000 non-null  int64         
 7   region_id               120000 non-null  int64         
 8   sales_rep_id            119270 non-null  float64       
 9   units                   120000 non-null  int64         
 10  asp_eur                 120000 non-null  float64       
 11  discount_pct            119047 non-null  float64       
 12  revenue_eur             120000

In [15]:
products = pd.read_sql_table(
    table_name="dimProduct",
    schema="staging",
    con=engine)

products.head()

,product_id,sku,product_family,product_group,product_name,launch_year,lifecycle_stage,base_list_price_eur,base_unit_cost_eur,target_margin_pct,product_growth_factor,is_declining_product,is_new_product
0,1000,SKU-1000,IT Devices,Laptop Pro,Laptop Pro Model 01,2022,Decline,804.37,540.92,0.29,1.08,1,0
1,1001,SKU-1001,IT Devices,Laptop Pro,Laptop Pro Model 02,2023,Growth,771.33,542.71,0.29,1.08,0,0
2,1002,SKU-1002,IT Devices,Laptop Pro,Laptop Pro Model 03,2024,Growth,828.80,552.00,0.29,1.08,0,0
3,1003,SKU-1003,IT Devices,Laptop Pro,Laptop Pro Model 04,2020,Mature,1137.76,781.22,0.29,1.08,0,0
4,1004,SKU-1004,IT Devices,Laptop Pro,Laptop Pro Model 05,2022,Mature,814.88,542.77,0.29,1.08,0,0


In [ ]:
customers = pd.read_sql_table(
    table_name="dimCustomer",
    schema="staging",
    con=engine)

customers.head(10)

,customer_id,customer_code,customer_segment,industry,region_id,customer_since,customer_size_score,base_churn_probability,churn_risk_band
0,1,CUST-00001,SMB,Retail,10,2020-08-12,1.127,0.106,Medium Churn Risk
1,2,CUST-00002,Public Sector,Retail,5,2021-10-04,1.699,0.068,Low Churn Risk
2,3,CUST-00003,Mid-Market,Manufacturing,6,2021-12-04,1.870,0.055,Low Churn Risk
3,4,CUST-00004,Mid-Market,Government,8,2024-01-15,1.847,0.072,Medium Churn Risk
4,5,CUST-00005,SMB,Retail,8,2022-08-26,0.937,0.199,High Churn Risk


In [ ]:
products["launch_date"] = pd.to_datetime(products["launch_year"].astype(str) + "-01-01", errors="coerce")
products.head()
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   product_id             200 non-null    int64         
 1   sku                    200 non-null    object        
 2   product_family         200 non-null    object        
 3   product_group          200 non-null    object        
 4   product_name           200 non-null    object        
 5   launch_year            200 non-null    int64         
 6   lifecycle_stage        200 non-null    object        
 7   base_list_price_eur    200 non-null    float64       
 8   base_unit_cost_eur     200 non-null    float64       
 9   target_margin_pct      200 non-null    float64       
 10  product_growth_factor  200 non-null    float64       
 11  is_declining_product   200 non-null    int64         
 12  is_new_product         200 non-null    int64         
 13  launc

In [28]:
customers.info()
sales = sales_transactions

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   customer_id             1200 non-null   int64         
 1   customer_code           1200 non-null   object        
 2   customer_segment        1200 non-null   object        
 3   industry                1200 non-null   object        
 4   region_id               1200 non-null   int64         
 5   customer_since          1200 non-null   datetime64[ns]
 6   customer_size_score     1200 non-null   float64       
 7   base_churn_probability  1200 non-null   float64       
 8   churn_risk_band         1200 non-null   object        
dtypes: datetime64[ns](1), float64(2), int64(2), object(4)
memory usage: 84.5+ KB


In [ ]:
# validation: run uniqueness checks
checks = {
    "duplicate_product_ids": int(products["product_id"].duplicated().sum()),
    "duplicate_customer_ids": int(customers["customer_id"].duplicated().sum()),
    "duplicate_transaction_ids": int(sales["sales_id"].duplicated().sum()),
    "sales_unknown_products": int((~sales["product_id"].isin(products["product_id"])).sum()),
    "sales_unknown_customers": int((~sales["customer_id"].isin(customers["customer_id"])).sum())}

checks

{'duplicate_product_ids': 0,
 'duplicate_customer_ids': 0,
 'duplicate_transaction_ids': 0,
 'sales_unknown_products': 0,
 'sales_unknown_customers': 0}

In [ ]:
# Helper function
# Cap extreme numeric values using IQR limits, optionally within groups
def cap_outliers_iqr(df: pd.DataFrame, columns: list[str], group_cols: list[str] | None = None, multiplier: float = 3.0) -> pd.DataFrame:
    out = df.copy()

    if group_cols is None:
        for col in columns:
            q1, q3 = out[col].quantile([0.25, 0.75])
            iqr = q3 - q1
            lower = q1 - multiplier * iqr
            upper = q3 + multiplier * iqr
            out[col] = out[col].clip(lower=lower, upper=upper)
        return out

    for col in columns:
        def _clip_group(s):
            q1, q3 = s.quantile([0.25, 0.75])
            iqr = q3 - q1
            return s.clip(lower=q1 - multiplier * iqr, upper=q3 + multiplier * iqr)

        out[col] = out.groupby(group_cols)[col].transform(_clip_group)

    return out

In [ ]:
    # Cap heavy-tailed business measures, but keep real zeros.
df = cap_outliers_iqr(
    sales,
    columns=["units", "revenue_eur", "website_visits", "demo_requests"],
    group_cols=["product_category", "region"],
    multiplier=4.0,)

In [31]:
customers_head = customers.head(10)
customers_head.to_csv("customers.csv")

In [32]:
products_head = products.head(10)
products_head.to_csv("products.csv")
sales_head = sales.head(10)
sales_head.to_csv("sales.csv")

In [33]:
SalesRep = pd.read_sql_table(
    table_name="dimSalesRep",
    schema="staging",
    con=engine)

SalesRep_head = SalesRep.head(10)
SalesRep_head 

,sales_rep_id,sales_rep_code,region_id,seniority,annual_quota_eur
0,1,REP-001,6,Professional,1395879.0
1,2,REP-002,7,Professional,1262654.0
2,3,REP-003,5,Junior,710781.0
3,4,REP-004,6,Senior,1952522.0
4,5,REP-005,10,Senior,2251696.0
5,6,REP-006,1,Senior,2810117.0
6,7,REP-007,8,Professional,1286526.0
7,8,REP-008,10,Senior,2490179.0
8,9,REP-009,3,Professional,1160163.0
9,10,REP-010,7,Professional,1217284.0


In [34]:
costs = pd.read_sql_table(
    table_name="factCosts",
    schema="staging",
    con=engine)

costs_head = costs.head(10)
costs_head 

,cost_id,year_month,product_id,standard_unit_cost_eur,actual_unit_cost_eur,cost_variance_eur,cost_variance_pct
0,1,2021-01-01,1000,541.10,573.85,32.75,0.060525
1,2,2021-01-01,1001,551.37,534.01,-17.36,-0.031485
2,3,2021-01-01,1002,555.77,541.92,-13.85,-0.024920
3,4,2021-01-01,1003,819.76,866.83,47.07,0.057419
4,5,2021-01-01,1004,544.95,577.78,32.83,0.060244
5,6,2021-01-01,1005,557.20,568.91,11.71,0.021016
6,7,2021-01-01,1006,625.80,646.56,20.76,0.033174
7,8,2021-01-01,1007,622.17,611.98,-10.19,-0.016378
8,9,2021-01-01,1008,549.48,577.21,27.73,0.050466
9,10,2021-01-01,1009,635.01,641.62,6.61,0.010409


In [35]:
returns = pd.read_sql_table(
    table_name="factReturns",
    schema="staging",
    con=engine)

returns_head = returns.head(10)
returns_head 

,return_id,sales_id,date_id,return_date,customer_id,product_id,region_id,return_units,return_value_eur,return_reason
0,1,71788,20250526,2025-05-26,928,1075,3,1,204.07,Defect
1,2,67219,20210611,2021-06-11,1077,1170,9,3,185.10,Customer Cancellation
2,3,54067,20211228,2021-12-28,379,1042,3,6,2003.92,Defect
3,4,7169,20220428,2022-04-28,533,1181,6,1,219.80,Other
4,5,29619,20221106,2022-11-06,928,1069,3,1,200.15,Defect
5,6,101426,20251012,2025-10-12,149,1139,8,20,1996.91,Wrong Configuration
6,7,20442,20251207,2025-12-07,984,1085,2,1,900.88,Customer Cancellation
7,8,2663,20250129,2025-01-29,894,1183,1,3,533.33,Shipping Damage
8,9,20372,20250123,2025-01-23,688,1075,9,4,819.84,Shipping Damage
9,10,108152,20220827,2022-08-27,469,1058,1,7,3478.21,Wrong Configuration


In [36]:
region = pd.read_sql_table(
    table_name="dimRegion",
    schema="staging",
    con=engine)

region_head = region.head(10)
region_head 

,region_id,country,region,currency,fx_to_eur,market_growth_factor,margin_factor
0,1,Germany,DACH,EUR,1.0000,1.06,0.98
1,2,France,Western Europe,EUR,1.0000,1.02,0.97
2,3,Spain,Southern Europe,EUR,1.0000,1.09,0.94
3,4,Italy,Southern Europe,EUR,1.0000,1.01,0.93
4,5,United Kingdom,Northern Europe,GBP,1.1700,0.97,1.01
5,6,United States,North America,USD,0.9200,1.14,1.03
6,7,Canada,North America,CAD,0.6800,1.04,0.99
7,8,Netherlands,Benelux,EUR,1.0000,1.07,1.00
8,9,Poland,Eastern Europe,PLN,0.2300,1.12,0.89
9,10,Japan,APAC,JPY,0.0062,0.95,1.02


In [37]:
inventory = pd.read_sql_table(
    table_name="factInventory",
    schema="staging",
    con=engine)

inventory_head = inventory.head(10)
inventory_head 

,inventory_id,year_month,product_id,region_id,opening_stock_units,production_units,ending_stock_units,stockout_flag,inventory_value_eur,zero_stock_flag
0,1,2021-01-01,1000,1,53,79,51,False,30108.39,0
1,2,2021-01-01,1000,3,31,22,34,False,11838.23,0
2,3,2021-01-01,1000,7,15,13,18,False,7186.07,0
3,4,2021-01-01,1001,1,3,4,1,False,551.01,0
4,5,2021-01-01,1001,2,180,118,196,False,43963.47,0
5,6,2021-01-01,1001,3,160,140,183,False,12511.12,0
6,7,2021-01-01,1001,5,7,7,7,False,2085.98,0
7,8,2021-01-01,1001,6,24,37,28,False,3674.63,0
8,9,2021-01-01,1001,8,94,72,91,False,18932.51,0
9,10,2021-01-01,1002,1,10,6,8,False,3978.41,0


In [38]:
crm = pd.read_sql_table(
    table_name="factCRMActivities",
    schema="staging",
    con=engine)

crm_head = crm.head(10)
crm_head 

,activity_id,date_id,activity_date,customer_id,sales_rep_id,activity_type,activity_minutes,sentiment_score,customer_health_score,customer_health_band
0,1,20241105,2024-11-05,1128,57,Call,22,1.386,90.3,Healthy
1,2,20221123,2022-11-23,1049,60,Email,40,-0.289,50.9,Neutral
2,3,20240619,2024-06-19,1006,48,Email,18,0.337,73.7,Neutral
3,4,20211028,2021-10-28,130,34,Call,28,-0.178,66.4,Neutral
4,5,20220722,2022-07-22,804,25,Visit,46,-0.347,50.6,Neutral
5,6,20210627,2021-06-27,1118,22,Call,19,0.312,77.4,Healthy
6,7,20220404,2022-04-04,1055,15,Call,53,0.875,70.9,Neutral
7,8,20221205,2022-12-05,1111,12,Call,8,-0.157,61.6,Neutral
8,9,20210922,2021-09-22,124,46,Email,44,0.062,56.1,Neutral
9,10,20240701,2024-07-01,153,11,Email,9,0.333,63.1,Neutral


In [39]:
pipeline = pd.read_sql_table(
    table_name="factPipeline",
    schema="staging",
    con=engine)

pipeline_head = pipeline.head(10)
pipeline_head

,opportunity_id,created_date,customer_id,product_group,sales_rep_id,stage,expected_value_eur,win_probability,expected_close_date,created_date_id,weighted_pipeline_eur,days_to_close,is_closed_won,is_closed_lost
0,1,2021-08-12,56,Connectivity Module,18,Lead,137697.63,0.413,2022-03-15,20210812,56869.12,215,0,0
1,2,2021-02-14,1038,Accessory Kit,41,Qualified,109985.57,0.452,2021-07-09,20210214,49713.48,145,0,0
2,3,2022-07-09,546,Service Contract,59,Qualified,31257.79,0.153,2022-12-06,20220709,4782.44,150,0,0
3,4,2025-08-05,1075,Legacy Workstation,67,Qualified,18770.05,0.699,2026-01-28,20250805,13120.26,176,0,0
4,5,2025-06-18,96,Monitoring Device,33,Lead,15327.18,0.425,2025-12-16,20250618,6514.05,181,0,0
5,6,2024-11-13,269,Accessory Kit,67,Qualified,17647.51,0.452,2025-02-23,20241113,7976.67,102,0,0
6,7,2025-06-18,1199,Legacy Workstation,37,Lead,33481.56,0.660,2026-01-02,20250618,22097.83,198,0,0
7,8,2023-09-20,164,Connectivity Module,47,Lead,23485.35,0.300,2024-04-02,20230920,7045.60,195,0,0
8,9,2022-09-27,860,Medical Sensor,70,Lead,9424.25,0.613,2023-02-23,20220927,5777.07,149,0,0
9,10,2025-09-06,977,Industrial Scanner,79,Qualified,19774.66,0.427,2026-03-23,20250906,8443.78,198,0,0


In [40]:
date = pd.read_sql_table(
    table_name="dimDate",
    schema="curated",
    con=engine)

date_head = date.head(10)
date_head 

,DateKey,Date,Year,Quarter,QuarterName,Month,MonthName,MonthShortName,YearMonth,YearMonthKey,MonthStartDate,MonthEndDate,Day,DayOfWeek,DayName,WeekOfYear,ISOWeek,IsWeekend
0,20200101,2020-01-01,2020,1,Q1,1,January,Jan,2020-01,202001,2020-01-01,2020-01-31,1,4,Wednesday,1,1,False
1,20200102,2020-01-02,2020,1,Q1,1,January,Jan,2020-01,202001,2020-01-01,2020-01-31,2,5,Thursday,1,1,False
2,20200103,2020-01-03,2020,1,Q1,1,January,Jan,2020-01,202001,2020-01-01,2020-01-31,3,6,Friday,1,1,False
3,20200104,2020-01-04,2020,1,Q1,1,January,Jan,2020-01,202001,2020-01-01,2020-01-31,4,7,Saturday,1,1,True
4,20200105,2020-01-05,2020,1,Q1,1,January,Jan,2020-01,202001,2020-01-01,2020-01-31,5,1,Sunday,2,1,True
5,20200106,2020-01-06,2020,1,Q1,1,January,Jan,2020-01,202001,2020-01-01,2020-01-31,6,2,Monday,2,2,False
6,20200107,2020-01-07,2020,1,Q1,1,January,Jan,2020-01,202001,2020-01-01,2020-01-31,7,3,Tuesday,2,2,False
7,20200108,2020-01-08,2020,1,Q1,1,January,Jan,2020-01,202001,2020-01-01,2020-01-31,8,4,Wednesday,2,2,False
8,20200109,2020-01-09,2020,1,Q1,1,January,Jan,2020-01,202001,2020-01-01,2020-01-31,9,5,Thursday,2,2,False
9,20200110,2020-01-10,2020,1,Q1,1,January,Jan,2020-01,202001,2020-01-01,2020-01-31,10,6,Friday,2,2,False


In [ ]:
costs.to_csv("costs.csv")
SalesRep.to_csv("SalesRep.csv")
returns.to_csv("returns.csv")
region.to_csv("region.csv")
crm.to_csv("crm.csv")
pipeline.to_csv("pipeline.csv")
date.to_csv("date.csv")
sales.to_csv("sales_transactions.csv")
products.to_csv("products.csv")
customers.to_csv("customers.csv")
inventory_head.to_csv("inventory.csv")

In [111]:
engine.dispose()

In [52]:
raw = {
    "sales": sales,
    "products": products,
    "customers": customers,
    "regions": region,
    "inventory": inventory,
    "costs": costs,
    "returns": returns,
    "crm": crm,
    "pipeline": pipeline,
    "sales_reps": SalesRep,
    "date": date}

In [53]:
# Validate the minimum fields needed to reproduce the old EWS table.
REQUIRED_SOURCE_COLUMNS = {
    "sales": ["order_date", "customer_id", "product_id", "region_id", "units", "revenue_eur", "discount_pct"],
    "products": ["product_id", "product_family", "product_group"],
    "regions": ["region_id", "region"],
    "inventory": ["year_month", "product_id", "region_id", "stockout_flag"],
}

source_issues = []
for table, columns in REQUIRED_SOURCE_COLUMNS.items():
    missing = [c for c in columns if c not in raw[table].columns]
    if missing:
        source_issues.append(f"{table}: {missing}")

if source_issues:
    raise ValueError("Missing required source columns -> " + " | ".join(source_issues))

print("Required source-column check passed.")

Required source-column check passed.


In [ ]:
def normalize_id(series):
    # Create stable join keys for integer, float-like and string IDs.
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .replace({"<NA>": pd.NA, "nan": pd.NA, "None": pd.NA})
    )

def to_month(series):
    return pd.to_datetime(series, errors="coerce").dt.to_period("M").dt.to_timestamp()

def numeric(df, column, default=np.nan):
    if column not in df.columns:
        return pd.Series(default, index=df.index, dtype="float64")
    return pd.to_numeric(df[column], errors="coerce")

def existing(df, columns):
    return [c for c in columns if c in df.columns]

# Dimensions
products = raw["products"].copy()
products["product_id"] = normalize_id(products["product_id"])
products["product_line"] = products["product_family"].fillna("Unknown")
products["product_category"] = products["product_group"].fillna("Unknown")
products["launch_date"] = pd.to_datetime(products.get("launch_date"), errors="coerce")
products = products.drop_duplicates("product_id")

regions = raw["regions"].copy()
regions["region_id"] = normalize_id(regions["region_id"])
regions = regions.drop_duplicates("region_id")

customers = raw["customers"].copy()
customers["customer_id"] = normalize_id(customers["customer_id"])
customers["region_id"] = normalize_id(customers["region_id"])
customers = customers.drop_duplicates("customer_id")

sales_reps = raw["sales_reps"].copy()
sales_reps["sales_rep_id"] = normalize_id(sales_reps["sales_rep_id"])
sales_reps["region_id"] = normalize_id(sales_reps["region_id"])
sales_reps = sales_reps.drop_duplicates("sales_rep_id")

product_dim_cols = existing(products, [
    "product_id", "product_line", "product_category", "product_name", "sku",
    "lifecycle_stage", "launch_date", "is_declining_product", "is_new_product",
    "base_list_price_eur", "base_unit_cost_eur", "target_margin_pct", "product_growth_factor"])
region_dim_cols = existing(regions, [
    "region_id", "region", "country", "currency", "fx_to_eur",
    "market_growth_factor", "margin_factor"])

print(f"Products: {products['product_id'].nunique():,}; regions: {regions['region_id'].nunique():,}; "
      f"customers: {customers['customer_id'].nunique():,}; sales reps: {sales_reps['sales_rep_id'].nunique():,}")

Products: 200; regions: 10; customers: 1,200; sales reps: 85


In [ ]:
# Canonical sales transaction table
sales = raw["sales"].copy()
for col in ["product_id", "customer_id", "region_id", "sales_rep_id"]:
    if col in sales.columns:
        sales[col] = normalize_id(sales[col])

sales["year_month"] = to_month(sales["year_month"] if "year_month" in sales else sales["order_date"])
sales["units_sold"] = numeric(sales, "units", 0).fillna(0)
sales["revenue"] = numeric(sales, "revenue_eur", 0).fillna(0)
sales["avg_discount_pct"] = numeric(sales, "discount_pct")
sales["transaction_id"] = (
    sales["sales_id"].astype("string") if "sales_id" in sales
    else sales["order_id"].astype("string"))

# Add canonical region and product hierarchy names
sales = sales.merge(regions[region_dim_cols], on="region_id", how="left", validate="m:1")
sales = sales.merge(products[product_dim_cols], on="product_id", how="left", validate="m:1")
sales["region"] = sales["region"].fillna("Unknown")
sales["product_line"] = sales["product_line"].fillna("Unknown")
sales["product_category"] = sales["product_category"].fillna("Unknown")

# Enrich transactions with rep attributes so the sales-rep file contributes to monthly features
rep_cols = existing(sales_reps, ["sales_rep_id", "seniority", "annual_quota_eur"])
sales = sales.merge(sales_reps[rep_cols], on="sales_rep_id", how="left", validate="m:1")

sales = sales.dropna(subset=["year_month", "product_id", "region_id"])
print(f"Canonical sales rows: {len(sales):,}; date range: "
      f"{sales['year_month'].min():%Y-%m} to {sales['year_month'].max():%Y-%m}")

Canonical sales rows: 120,000; date range: 2021-01 to 2025-12


In [ ]:
# Customer-product-month table
customer_product_monthly = (
    sales.groupby(["year_month", "customer_id", "product_id"], dropna=False, as_index=False)
    .agg(
        units_sold=("units_sold", "sum"),
        revenue=("revenue", "sum"),
        order_count=("transaction_id", "nunique"),
        avg_discount_pct=("avg_discount_pct", "mean")))
customer_product_monthly["year_month"] = customer_product_monthly["year_month"].dt.strftime("%Y-%m")
customer_product_monthly["revenue"] = customer_product_monthly["revenue"].round(2)
customer_product_monthly["avg_discount_pct"] = customer_product_monthly["avg_discount_pct"].round(4)

monthly_sales = (
    sales.groupby(["product_id", "year_month", "region"], as_index=False)
    .agg(
        units_sold=("units_sold", "sum"),
        revenue=("revenue", "sum"),
        unique_customers=("customer_id", "nunique"),
        avg_discount_pct=("avg_discount_pct", "mean"),
        order_count=("transaction_id", "nunique"),
        active_sales_reps=("sales_rep_id", "nunique"),
        covered_annual_quota_eur=("annual_quota_eur", "sum"),
        gross_profit_eur=("gross_profit_eur", "sum") if "gross_profit_eur" in sales else ("revenue", "sum"),
        cogs_eur=("cogs_eur", "sum") if "cogs_eur" in sales else ("revenue", "sum")))

In [ ]:
# Build a complete product-month-region panel
# Pre-launch months are excluded where launch dates exist
date_candidates = [sales["year_month"]]
for table, col in [(raw["inventory"], "year_month"), (raw["costs"], "year_month")]:
    if col in table:
        date_candidates.append(to_month(table[col]))

all_dates = pd.concat(date_candidates, ignore_index=True).dropna()
min_month, max_month = all_dates.min(), all_dates.max()
months = pd.date_range(min_month, max_month, freq="MS")

# Use all IDs found in the product dimension or the sales fact
product_ids = pd.Index(products["product_id"].dropna().unique()).union(
    pd.Index(sales["product_id"].dropna().unique()))
region_keys = pd.Index(regions["region"].dropna().unique())

panel = (
    pd.MultiIndex.from_product(
        [product_ids, months, region_keys],
        names=["product_id", "year_month", "region"])
    .to_frame(index=False)
    .merge(products[product_dim_cols], on="product_id", how="left", validate="m:1"))
panel["product_line"] = panel["product_line"].fillna("Unknown")
panel["product_category"] = panel["product_category"].fillna("Unknown")

if "launch_date" in panel:
    keep = panel["launch_date"].isna() | (panel["year_month"] >= panel["launch_date"].dt.to_period("M").dt.to_timestamp())
    panel = panel.loc[keep].copy()

out = panel.merge(
    monthly_sales,
    on=["product_id", "year_month", "region"],
    how="left",
    validate="1:1")
zero_sales_cols = [
    "units_sold", "revenue", "unique_customers", "avg_discount_pct", "order_count",
    "active_sales_reps", "covered_annual_quota_eur", "gross_profit_eur", "cogs_eur"]
out[zero_sales_cols] = out[zero_sales_cols].fillna(0)
print(f"Complete panel: {len(out):,} product-month-region rows ({len(months)} months)")

Complete panel: 78,816 product-month-region rows (60 months)


In [ ]:
# Inventory and backorder proxy
inventory = raw["inventory"].copy()
inventory["product_id"] = normalize_id(inventory["product_id"])
inventory["region_id"] = normalize_id(inventory["region_id"])
inventory["year_month"] = to_month(inventory["year_month"])
inventory["stockout_flag"] = numeric(inventory, "stockout_flag", 0).fillna(0).astype(int)
inventory = inventory.merge(regions[["region_id", "region"]], on="region_id", how="left", validate="m:1")

inventory_monthly = (
    inventory.groupby(["product_id", "year_month", "region"], as_index=False)
    .agg(
        opening_stock_units=("opening_stock_units", "sum"),
        production_units=("production_units", "sum"),
        ending_stock_units=("ending_stock_units", "sum"),
        stockout_flag=("stockout_flag", "max"),
        zero_stock_flag=("zero_stock_flag", "max"),
        inventory_value_eur=("inventory_value_eur", "sum")))
out = out.merge(inventory_monthly, on=["product_id", "year_month", "region"], how="left", validate="1:1")
inventory_available = out["opening_stock_units"].notna() | out["production_units"].notna()
supply = out["opening_stock_units"].fillna(0) + out["production_units"].fillna(0)
out["backorder_units"] = np.where(inventory_available, (out["units_sold"] - supply).clip(lower=0), 0)
out["stockout_flag"] = out["stockout_flag"].fillna(0).astype(int)

In [59]:
# Costs, returns and the date dimension provide additional current-data features.
costs = raw["costs"].copy()
costs["product_id"] = normalize_id(costs["product_id"])
costs["year_month"] = to_month(costs["year_month"])
cost_monthly = (
    costs.groupby(["product_id", "year_month"], as_index=False)
    .agg(
        standard_unit_cost_eur=("standard_unit_cost_eur", "mean"),
        actual_unit_cost_eur=("actual_unit_cost_eur", "mean"),
        cost_variance_eur=("cost_variance_eur", "mean"),
        cost_variance_pct=("cost_variance_pct", "mean")))
out = out.merge(cost_monthly, on=["product_id", "year_month"], how="left", validate="m:1")

returns = raw["returns"].copy()
returns["product_id"] = normalize_id(returns["product_id"])
returns["region_id"] = normalize_id(returns["region_id"])
returns["year_month"] = to_month(returns["return_date"])
returns = returns.merge(regions[["region_id", "region"]], on="region_id", how="left", validate="m:1")
returns_monthly = (
    returns.groupby(["product_id", "year_month", "region"], as_index=False)
    .agg(return_units=("return_units", "sum"), return_value_eur=("return_value_eur", "sum"), return_count=("return_id", "nunique"))
)
out = out.merge(returns_monthly, on=["product_id", "year_month", "region"], how="left", validate="1:1")
out[["return_units", "return_value_eur", "return_count"]] = out[["return_units", "return_value_eur", "return_count"]].fillna(0)
out["return_rate_units"] = out["return_units"] / out["units_sold"].replace(0, np.nan)

dates = raw["date"].copy()
dates["year_month"] = to_month(dates["MonthStartDate"] if "MonthStartDate" in dates else dates["Date"])
date_monthly = dates.sort_values("year_month").drop_duplicates("year_month")
calendar_cols = existing(date_monthly, ["year_month", "Year", "Quarter", "QuarterName", "Month", "MonthName", "YearMonthKey"])
out = out.merge(date_monthly[calendar_cols], on="year_month", how="left", validate="m:1")

In [ ]:
# CRM engagement by region-month
crm = raw["crm"].copy()
crm["customer_id"] = normalize_id(crm["customer_id"])
crm["year_month"] = to_month(crm["activity_date"])
crm = crm.merge(customers[["customer_id", "region_id"]], on="customer_id", how="left", validate="m:1")
crm = crm.merge(regions[["region_id", "region"]], on="region_id", how="left", validate="m:1")
crm["is_demo_like"] = crm["activity_type"].astype("string").str.contains(
    "demo|visit|meeting|presentation", case=False, na=False
).astype(int)
crm_region_month = (
    crm.dropna(subset=["year_month", "region"])
    .groupby(["year_month", "region"], as_index=False)
    .agg(
        crm_activity_count=("activity_id", "nunique"),
        crm_activity_minutes=("activity_minutes", "sum"),
        crm_demo_like_count=("is_demo_like", "sum"),
        avg_sentiment_score=("sentiment_score", "mean"),
        avg_customer_health_score=("customer_health_score", "mean")))
out = out.merge(crm_region_month, on=["year_month", "region"], how="left", validate="m:1")

# Pipeline engagement by product category, customer region and creation month
pipeline = raw["pipeline"].copy()
pipeline["customer_id"] = normalize_id(pipeline["customer_id"])
pipeline["year_month"] = to_month(pipeline["created_date"])
pipeline["product_category"] = pipeline["product_group"].fillna("Unknown")
pipeline = pipeline.merge(customers[["customer_id", "region_id"]], on="customer_id", how="left", validate="m:1")
pipeline = pipeline.merge(regions[["region_id", "region"]], on="region_id", how="left", validate="m:1")
pipeline_monthly = (
    pipeline.dropna(subset=["year_month", "region"])
    .groupby(["year_month", "region", "product_category"], as_index=False)
    .agg(
        pipeline_opportunities=("opportunity_id", "nunique"),
        weighted_pipeline_eur=("weighted_pipeline_eur", "sum"),
        expected_pipeline_eur=("expected_value_eur", "sum"),
        avg_win_probability=("win_probability", "mean"),
        closed_won_count=("is_closed_won", "sum"),
        closed_lost_count=("is_closed_lost", "sum")))
out = out.merge(
    pipeline_monthly,
    on=["year_month", "region", "product_category"],
    how="left",
    validate="m:1")

activity_zero_cols = [
    "crm_activity_count", "crm_activity_minutes", "crm_demo_like_count",
    "pipeline_opportunities", "weighted_pipeline_eur", "expected_pipeline_eur",
    "closed_won_count", "closed_lost_count"]
out[activity_zero_cols] = out[activity_zero_cols].fillna(0)
out["campaign_flag"] = ((out["crm_activity_count"] > 0) | (out["pipeline_opportunities"] > 0)).astype(int)
out["website_visits"] = out["crm_activity_count"].astype(int)
out["demo_requests"] = (out["crm_demo_like_count"] + out["pipeline_opportunities"]).astype(int)

In [61]:
# Derive market-demand and competition proxies from actual current sales
market = (
    out.groupby(["product_category", "region", "year_month"], as_index=False)
    .agg(category_region_units=("units_sold", "sum"))
    .sort_values(["product_category", "region", "year_month"]))
market_group = market.groupby(["product_category", "region"], group_keys=False)
market["market_units_baseline_6m"] = market_group["category_region_units"].transform(
    lambda s: s.shift(1).rolling(6, min_periods=1).mean())
ratio = market["category_region_units"] / market["market_units_baseline_6m"].replace(0, np.nan)
market["market_demand_index"] = (ratio * 100).clip(0, 300)
no_history = market["market_units_baseline_6m"].isna()
market.loc[no_history & (market["category_region_units"] == 0), "market_demand_index"] = 100
market.loc[no_history & (market["category_region_units"] > 0), "market_demand_index"] = 100

out = out.merge(
    market[["product_category", "region", "year_month", "category_region_units", "market_units_baseline_6m", "market_demand_index"]],
    on=["product_category", "region", "year_month"],
    how="left",
    validate="m:1")
product_share = out["units_sold"] / out["category_region_units"].replace(0, np.nan)
out["competitor_pressure_index"] = (100 * (1 - product_share)).clip(0, 100)
out.loc[out["category_region_units"].eq(0), "competitor_pressure_index"] = 100

print("Proxy fields created from observed sales, CRM and pipeline data (no random values).")

Proxy fields created from observed sales, CRM and pipeline data (no random values).


In [63]:
def slope(values):
    values = np.asarray(values, dtype=float)
    if len(values) < 2 or np.all(np.isnan(values)):
        return np.nan
    return float(np.polyfit(np.arange(len(values)), values, 1)[0])

out = out.sort_values(["product_id", "region", "year_month"]).reset_index(drop=True)
group = out.groupby(["product_id", "region"], group_keys=False)

# Leakage-aware predictors: rolling statistics use only months before the current row.
out["units_lag_1m"] = group["units_sold"].shift(1)
out["units_lag_3m"] = group["units_sold"].shift(3)
out["revenue_lag_1m"] = group["revenue"].shift(1)
out["customers_lag_1m"] = group["unique_customers"].shift(1)

out["rolling_units_mean_3m"] = group["units_sold"].transform(
    lambda s: s.shift(1).rolling(3, min_periods=2).mean())
out["rolling_units_mean_6m"] = group["units_sold"].transform(
    lambda s: s.shift(1).rolling(6, min_periods=3).mean())
out["rolling_units_std_3m"] = group["units_sold"].transform(
    lambda s: s.shift(1).rolling(3, min_periods=2).std())
out["volatility_3m"] = out["rolling_units_std_3m"] / out["rolling_units_mean_3m"].replace(0, np.nan)

out["mom_units_growth_pct"] = (
    (out["units_sold"] - out["units_lag_1m"]) / out["units_lag_1m"].replace(0, np.nan))
out["customer_count_growth_pct"] = (
    (out["unique_customers"] - out["customers_lag_1m"]) / out["customers_lag_1m"].replace(0, np.nan))
out["trend_slope_3m"] = group["units_sold"].transform(
    lambda s: s.shift(1).rolling(3, min_periods=3).apply(slope, raw=True))
out["trend_slope_6m"] = group["units_sold"].transform(
    lambda s: s.shift(1).rolling(6, min_periods=6).apply(slope, raw=True))

expected_units = out["rolling_units_mean_3m"] + out["trend_slope_3m"].fillna(0)
out["risk_factor_sales_drop"] = (out["mom_units_growth_pct"] < -0.25).astype(int)
out["risk_factor_customer_drop"] = (out["customer_count_growth_pct"] < -0.25).astype(int)
out["risk_factor_volatility"] = (out["volatility_3m"] > 0.45).astype(int)
out["risk_factor_under_trend"] = (
    (out["units_sold"] < expected_units * 0.75) & (expected_units > 0)
).astype(int)

In [ ]:
# One-month-ahead target
# Future columns are temporary and are not exported as model features
out["future_units"] = group["units_sold"].shift(-1)
out["future_customers"] = group["unique_customers"].shift(-1)
out["future_volatility"] = group["volatility_3m"].shift(-1)
out["future_expected_units"] = (
    group["rolling_units_mean_3m"].shift(-1)
    + group["trend_slope_3m"].shift(-1).fillna(0))

future_sales_drop = (out["future_units"] < out["units_sold"] * 0.75).astype(int)
future_customer_drop = (out["future_customers"] < out["unique_customers"] * 0.75).astype(int)
future_high_volatility = (out["future_volatility"] > 0.50).astype(int)
future_under_trend = (
    (out["future_units"] < out["future_expected_units"] * 0.72)
    & (out["future_expected_units"] > 0)
).astype(int)
future_stockout = group["stockout_flag"].shift(-1).fillna(0).astype(int)

risk_score = (
    future_sales_drop + future_customer_drop + future_high_volatility
    + future_under_trend + future_stockout)
out["next_month_risk_label"] = np.select(
    [risk_score >= 3, risk_score >= 1], [2, 1], default=0
).astype(float)

# Every group's final observation has no known next month
last_row = group.cumcount(ascending=False).eq(0)
out.loc[last_row, "next_month_risk_label"] = np.nan

In [ ]:
#final schema 
EWS_COLUMNS = [
    "product_id", "year_month", "product_line", "product_category", "region",
    "units_sold", "revenue", "unique_customers", "avg_discount_pct",
    "units_lag_1m", "units_lag_3m", "revenue_lag_1m",
    "rolling_units_mean_3m", "rolling_units_mean_6m", "rolling_units_std_3m",
    "volatility_3m", "mom_units_growth_pct", "customer_count_growth_pct",
    "trend_slope_3m", "trend_slope_6m", "stockout_flag", "backorder_units",
    "market_demand_index", "competitor_pressure_index", "campaign_flag",
    "website_visits", "demo_requests", "risk_factor_sales_drop",
    "risk_factor_customer_drop", "risk_factor_volatility",
    "risk_factor_under_trend", "next_month_risk_label"]

missing_final = [c for c in EWS_COLUMNS if c not in out.columns]
if missing_final:
    raise AssertionError(f"Old EWS columns missing: {missing_final}")

final_modeling_dataset = out[EWS_COLUMNS].copy()
final_modeling_dataset["year_month"] = final_modeling_dataset["year_month"].dt.strftime("%Y-%m")
numeric_cols = final_modeling_dataset.select_dtypes(include=[np.number]).columns
final_modeling_dataset[numeric_cols] = final_modeling_dataset[numeric_cols].replace([np.inf, -np.inf], np.nan)

# Enriched version keeps every old column first, followed by current-source fields
temporary_future_cols = ["future_units", "future_customers", "future_volatility", "future_expected_units"]
enriched_extra_cols = [
    c for c in out.columns
    if c not in EWS_COLUMNS and c not in temporary_future_cols
]
enriched_ews_modeling_dataset = out[EWS_COLUMNS + enriched_extra_cols].copy()
enriched_ews_modeling_dataset["year_month"] = enriched_ews_modeling_dataset["year_month"].dt.strftime("%Y-%m")

print(f"Final EWS dataset: {len(final_modeling_dataset):,} rows x {final_modeling_dataset.shape[1]} columns")
print("Missing original columns:", [c for c in EWS_COLUMNS if c not in final_modeling_dataset])

Final EWS dataset: 78,816 rows x 32 columns
Missing original columns: []


In [ ]:
# Column dictionary
definitions = {
    "product_id": "Unique product key.",
    "year_month": "Calendar month in YYYY-MM format.",
    "product_line": "Current product_family mapped to the old product_line field.",
    "product_category": "Current product_group mapped to the old product_category field.",
    "region": "Geographical region from the region dimension.",
    "units_sold": "Units sold in the product-region-month.",
    "revenue": "Revenue in EUR in the product-region-month.",
    "unique_customers": "Distinct purchasing customers.",
    "avg_discount_pct": "Mean transaction discount percentage.",
    "units_lag_1m": "Units sold one month earlier.",
    "units_lag_3m": "Units sold three months earlier.",
    "revenue_lag_1m": "Revenue one month earlier.",
    "rolling_units_mean_3m": "Mean units over the prior three months; current month excluded.",
    "rolling_units_mean_6m": "Mean units over the prior six months; current month excluded.",
    "rolling_units_std_3m": "Unit standard deviation over the prior three months.",
    "volatility_3m": "Prior-three-month standard deviation divided by prior-three-month mean.",
    "mom_units_growth_pct": "Current units versus previous-month units.",
    "customer_count_growth_pct": "Current unique customers versus previous month.",
    "trend_slope_3m": "Linear unit trend across the prior three months.",
    "trend_slope_6m": "Linear unit trend across the prior six months.",
    "stockout_flag": "Maximum stockout flag from inventory for the month.",
    "backorder_units": "Proxy: units above opening stock plus production; zero if inventory unavailable.",
    "market_demand_index": "Proxy: category-region units versus trailing six-month baseline, indexed to 100.",
    "competitor_pressure_index": "Proxy: 100 minus the product unit share within category-region-month.",
    "campaign_flag": "Proxy: one when CRM or matching pipeline activity exists.",
    "website_visits": "Proxy: regional CRM activity count; not literal website visits.",
    "demo_requests": "Proxy: demo/visit/meeting CRM activities plus new opportunities.",
    "risk_factor_sales_drop": "One when month-over-month unit growth is below -25%.",
    "risk_factor_customer_drop": "One when customer-count growth is below -25%.",
    "risk_factor_volatility": "One when three-month volatility exceeds 0.45.",
    "risk_factor_under_trend": "One when units are below 75% of trend-adjusted expectation.",
    "next_month_risk_label": "Target: 0 low, 1 medium, 2 high based on next-month risk triggers; unavailable for final month.",
}
ews_column_dictionary = pd.DataFrame({
    "column": EWS_COLUMNS,
    "definition": [definitions[c] for c in EWS_COLUMNS]})
display(ews_column_dictionary)

,column,definition
0,product_id,Unique product key.
1,year_month,Calendar month in YYYY-MM format.
2,product_line,Current product_family mapped to the old produ...
3,product_category,Current product_group mapped to the old produc...
4,region,Geographical region from the region dimension.
5,units_sold,Units sold in the product-region-month.
6,revenue,Revenue in EUR in the product-region-month.
7,unique_customers,Distinct purchasing customers.
8,avg_discount_pct,Mean transaction discount percentage.
9,units_lag_1m,Units sold one month earlier.


In [ ]:
# Final QA checks
checks = {
    "all_32_old_columns_present": list(final_modeling_dataset.columns) == EWS_COLUMNS,
    "unique_product_month_region_key": not final_modeling_dataset.duplicated(
        ["product_id", "year_month", "region"]
    ).any(),
    "no_future_helper_columns_exported": not any(
        c.startswith("future_") for c in final_modeling_dataset.columns
    ),
    "last_row_target_is_missing": final_modeling_dataset.groupby(
        ["product_id", "region"], dropna=False
    ).tail(1)["next_month_risk_label"].isna().all(),
    "risk_factors_are_binary": all(
        set(final_modeling_dataset[c].dropna().unique()).issubset({0, 1})
        for c in [
            "risk_factor_sales_drop", "risk_factor_customer_drop",
            "risk_factor_volatility", "risk_factor_under_trend", "stockout_flag"])}

qa = pd.Series(checks, name="passed").rename_axis("check").reset_index()
display(qa)
if not all(checks.values()):
    raise AssertionError("At least one final QA check failed.")

print("\nTarget distribution (excluding rows without a future month):")
print(final_modeling_dataset["next_month_risk_label"].value_counts(dropna=False).sort_index())

,check,passed
0,all_32_old_columns_present,True
1,unique_product_month_region_key,True
2,no_future_helper_columns_exported,True
3,last_row_target_is_missing,True
4,risk_factors_are_binary,True



Target distribution (excluding rows without a future month):
next_month_risk_label
0.0    10714
1.0    40270
2.0    26232
NaN     1600
Name: count, dtype: int64


In [79]:
# Save all deliverables
final_modeling_dataset.to_csv(OUTPUT_DIR_FEATURE_ENGINEERING / "ews_modeling_dataset.csv", index=False)
enriched_ews_modeling_dataset.to_csv(OUTPUT_DIR_FEATURE_ENGINEERING / "ews_modeling_dataset_enriched.csv", index=False)
customer_product_monthly.to_csv(OUTPUT_DIR_FEATURE_ENGINEERING / "customer_product_monthly.csv", index=False)
ews_column_dictionary.to_csv(OUTPUT_DIR_FEATURE_ENGINEERING / "ews_column_dictionary.csv", index=False)

print("Created:")
for path in sorted(OUTPUT_DIR_FEATURE_ENGINEERING.glob("*.csv")):
    print(f"- {path} ({path.stat().st_size:,} bytes)")

Created:
- ..\data\feature engineering\customer_product_monthly.csv (4,416,676 bytes)
- ..\data\feature engineering\ews_column_dictionary.csv (2,280 bytes)
- ..\data\feature engineering\ews_modeling_dataset.csv (20,054,521 bytes)
- ..\data\feature engineering\ews_modeling_dataset_enriched.csv (45,807,124 bytes)


In [77]:
TARGET_COL = "next_month_risk_label"
ID_COLS = ["product_id", "year_month", "region"]
CATEGORICAL_FEATURES = ["product_line", "product_category", "region"]

NUMERIC_FEATURES = [
    "units_sold",
    "revenue",
    "unique_customers",
    "avg_discount_pct",
    "units_lag_1m",
    "units_lag_3m",
    "revenue_lag_1m",
    "rolling_units_mean_3m",
    "rolling_units_mean_6m",
    "rolling_units_std_3m",
    "volatility_3m",
    "mom_units_growth_pct",
    "customer_count_growth_pct",
    "trend_slope_3m",
    "trend_slope_6m",
    "stockout_flag",
    "backorder_units",
    "market_demand_index",
    "competitor_pressure_index",
    "campaign_flag",
    "website_visits",
    "demo_requests",
    "risk_factor_sales_drop",
    "risk_factor_customer_drop",
    "risk_factor_volatility",
    "risk_factor_under_trend",
]

In [80]:
# helper preprocesser
# Fit simple preprocessing statistics on train only
def fit_pandas_preprocessor(train: pd.DataFrame, numeric_features: list[str], categorical_features: list[str]) -> dict[str, object]:
    numeric_medians = train[numeric_features].median(numeric_only=True).fillna(0)
    numeric_means = train[numeric_features].fillna(numeric_medians).mean()
    numeric_stds = train[numeric_features].fillna(numeric_medians).std().replace(0, 1).fillna(1)

    categorical_modes = {}
    categorical_levels = {}
    for col in categorical_features:
        mode = train[col].mode(dropna=True)
        categorical_modes[col] = mode.iloc[0] if len(mode) else "Unknown"
        values = train[col].fillna(categorical_modes[col]).astype(str)
        categorical_levels[col] = sorted(values.unique().tolist())

    return {
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "numeric_medians": numeric_medians,
        "numeric_means": numeric_means,
        "numeric_stds": numeric_stds,
        "categorical_modes": categorical_modes,
        "categorical_levels": categorical_levels}

# Apply train-fitted imputation, scaling, and one-hot encoding
def transform_with_pandas_preprocessor(df: pd.DataFrame, preprocessor: dict[str, object]) -> pd.DataFrame:

    numeric_features = preprocessor["numeric_features"]
    categorical_features = preprocessor["categorical_features"]

    numeric = df[numeric_features].copy()
    numeric = numeric.fillna(preprocessor["numeric_medians"])
    numeric = (numeric - preprocessor["numeric_means"]) / preprocessor["numeric_stds"]

    encoded_parts = [numeric.reset_index(drop=True)]
    for col in categorical_features:
        values = df[col].fillna(preprocessor["categorical_modes"][col]).astype(str)
        levels = preprocessor["categorical_levels"][col]
        encoded = pd.DataFrame(
            {f"{col}__{level}": (values == level).astype(int).to_numpy() for level in levels}
        )
        encoded_parts.append(encoded)

    return pd.concat(encoded_parts, axis=1)

# Create train/validation/test splits by month to respect forecasting chronology
def create_time_based_splits(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    
    labeled = df.dropna(subset=[TARGET_COL]).copy()
    labeled[TARGET_COL] = labeled[TARGET_COL].astype(int)

    months = sorted(labeled["year_month"].unique())
    if len(months) < 12:
        raise ValueError("Need at least 12 labeled months for robust time-based splits.")

    train_end = months[int(len(months) * 0.70)]
    valid_end = months[int(len(months) * 0.85)]

    train = labeled[labeled["year_month"] <= train_end].copy()
    valid = labeled[(labeled["year_month"] > train_end) & (labeled["year_month"] <= valid_end)].copy()
    test = labeled[labeled["year_month"] > valid_end].copy()

    return train, valid, test


# Fit preprocessing on train only, then transform validation and test
def transform_and_save_ml_matrices(
    train: pd.DataFrame,
    valid: pd.DataFrame,
    test: pd.DataFrame,
    numeric_features: list[str],
    categorical_features: list[str],
) -> dict[str, object]:
    preprocessor = fit_pandas_preprocessor(train, numeric_features, categorical_features)

    train_ml = transform_with_pandas_preprocessor(train, preprocessor)
    valid_ml = transform_with_pandas_preprocessor(valid, preprocessor)
    test_ml = transform_with_pandas_preprocessor(test, preprocessor)

    feature_names = train_ml.columns.tolist()

    train_ml[TARGET_COL] = train[TARGET_COL].to_numpy()
    valid_ml[TARGET_COL] = valid[TARGET_COL].to_numpy()
    test_ml[TARGET_COL] = test[TARGET_COL].to_numpy()

    train_ml.to_csv(OUTPUT_DIR_PREPROCESSED / "train_ml_ready.csv", index=False)
    valid_ml.to_csv(OUTPUT_DIR_PREPROCESSED / "valid_ml_ready.csv", index=False)
    test_ml.to_csv(OUTPUT_DIR_PREPROCESSED / "test_ml_ready.csv", index=False)

    train[ID_COLS + [TARGET_COL]].to_csv(OUTPUT_DIR_PREPROCESSED / "train_ids.csv", index=False)
    valid[ID_COLS + [TARGET_COL]].to_csv(OUTPUT_DIR_PREPROCESSED / "valid_ids.csv", index=False)
    test[ID_COLS + [TARGET_COL]].to_csv(OUTPUT_DIR_PREPROCESSED / "test_ids.csv", index=False)

    return {
        "feature_count": len(feature_names),
        "feature_names": feature_names,
        "train_shape": train_ml.shape,
        "valid_shape": valid_ml.shape,
        "test_shape": test_ml.shape}

In [87]:
# Collect compact QA metadata for reproducibility and review
def create_quality_report(
    raw_datasets: dict[str, pd.DataFrame],
    cleaned_modeling: pd.DataFrame,
    key_checks: dict[str, object],
    split_summary: dict[str, object],
    numeric_features: list[str],
) -> dict[str, object]:
    report = {
        "raw_shapes": {name: list(df.shape) for name, df in raw_datasets.items()},
        "key_checks": key_checks,
        "cleaned_modeling_shape": list(cleaned_modeling.shape),
        "target_distribution": (
            cleaned_modeling[TARGET_COL]
            .value_counts(dropna=False)
            .sort_index()
            .astype(int)
            .to_dict()),
        "missing_rate_top_15": (
            cleaned_modeling[numeric_features + CATEGORICAL_FEATURES + [TARGET_COL]]
            .isna()
            .mean()
            .sort_values(ascending=False)
            .head(15)
            .round(4)
            .to_dict()),
        "split_summary": {
            "feature_count": split_summary["feature_count"],
            "train_shape": list(split_summary["train_shape"]),
            "valid_shape": list(split_summary["valid_shape"]),
            "test_shape": list(split_summary["test_shape"])}}

    with open(OUTPUT_DIR_PREPROCESSED / "preprocessing_quality_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, default=str)

    return report

In [90]:
# Load all expected EWS CSV files into a dictionary
def load_csv_files(data_dir: Path = OUTPUT_DIR_FEATURE_ENGINEERING) -> dict[str, pd.DataFrame]:
    expected_files = [
        "sales_transactions",
        "products",
        "customers",
        "customer_product_monthly",
        "inventory",
        "ews_modeling_dataset_enriched"]

    datasets = {}
    for name in expected_files:
        path = data_dir / f"{name}.csv"
        if not path.exists():
            raise FileNotFoundError(f"Missing required file: {path}")
        datasets[name] = pd.read_csv(path)

    return datasets

In [99]:
# Convert date/month columns to consistent datetime-friendly representations
def standardize_dates(datasets: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    datasets = {name: df.copy() for name, df in datasets.items()}
    if "order_date" in datasets["sales_transactions"]:
        datasets["sales_transactions"]["order_date"] = pd.to_datetime(
            datasets["sales_transactions"]["order_date"], errors="coerce")
    for name in [
        "customer_product_monthly",
        "inventory",
        "ews_modeling_dataset_enriched"]:
        datasets[name]["year_month"] = pd.to_datetime(
            datasets[name]["year_month"].astype(str) + "-01", errors="coerce")

    datasets["products"]["launch_date"] = pd.to_datetime(
        datasets["products"]["launch_date"], errors="coerce")
    datasets["customers"]["customer_since"] = pd.to_datetime(
        datasets["customers"]["customer_since"], errors="coerce")

    return datasets

In [92]:
# Run relationship and uniqueness checks across the synthetic relational data
def validate_keys(datasets: dict[str, pd.DataFrame]) -> dict[str, object]:
    sales = datasets["sales_transactions"]
    products = datasets["products"]
    customers = datasets["customers"]
    final = datasets["ews_modeling_dataset_enriched"]

    checks = {
        "duplicate_product_ids": int(products["product_id"].duplicated().sum()),
        "duplicate_customer_ids": int(customers["customer_id"].duplicated().sum()),
        "duplicate_transaction_ids": int(sales["transaction_id"].duplicated().sum()),
        "sales_unknown_products": int((~sales["product_id"].isin(products["product_id"])).sum()),
        "sales_unknown_customers": int((~sales["customer_id"].isin(customers["customer_id"])).sum()),
        "final_unknown_products": int((~final["product_id"].isin(products["product_id"])).sum()),
        "final_duplicate_panel_rows": int(final.duplicated(["product_id", "year_month", "region"]).sum())    }

    return checks

In [106]:
# Clean product/customer dimension tables without changing their business meaning
def clean_dimension_tables(datasets: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    datasets = {name: df.copy() for name, df in datasets.items()}

    products = datasets["products"]
#    products["is_strategic_product"] = products["is_strategic_product"].astype(str).str.lower().isin(["true", "1", "yes"])
#    products["target_margin_pct"] = products["target_margin_pct"].clip(0, 1)
#    products["average_unit_cost"] = products["average_unit_cost"].clip(lower=0)
    products = products.drop_duplicates("product_id").reset_index(drop=True)
    datasets["products"] = products

    customers = datasets["customers"]
#     customers["account_status"] = customers["account_status"].fillna("Unknown")
    customers["customer_segment"] = customers["customer_segment"].fillna("Unknown")
    customers = customers.drop_duplicates("customer_id").reset_index(drop=True)
    datasets["customers"] = customers

    return datasets

In [108]:
# Clean transactional data and preserve a row-level audit trail
def clean_transaction_table(datasets: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    datasets = {name: df.copy() for name, df in datasets.items()}
    sales = datasets["sales_transactions"]

    sales = sales.drop_duplicates("transaction_id").copy()
    sales["units_sold"] = pd.to_numeric(sales["units_sold"], errors="coerce")
    sales["revenue"] = pd.to_numeric(sales["revenue"], errors="coerce")
    sales["discount_pct"] = pd.to_numeric(sales["discount_pct"], errors="coerce")

    sales["units_sold"] = sales["units_sold"].clip(lower=0)
    sales["discount_pct"] = sales["discount_pct"].clip(lower=0, upper=0.60)
    sales["sales_rep_id"] = sales["sales_rep_id"].fillna("Unknown")
    sales["country"] = sales["country"].fillna("Unknown")

    datasets["sales_transactions"] = sales.reset_index(drop=True)
    return datasets

In [94]:
# Cap extreme numeric values using IQR limits, optionally within groups
def cap_outliers_iqr(df: pd.DataFrame, columns: list[str], group_cols: list[str] | None = None, multiplier: float = 3.0) -> pd.DataFrame:
    out = df.copy()
    if group_cols is None:
        for col in columns:
            q1, q3 = out[col].quantile([0.25, 0.75])
            iqr = q3 - q1
            lower = q1 - multiplier * iqr
            upper = q3 + multiplier * iqr
            out[col] = out[col].clip(lower=lower, upper=upper)
        return out

    for col in columns:
        def _clip_group(s):
            q1, q3 = s.quantile([0.25, 0.75])
            iqr = q3 - q1
            return s.clip(lower=q1 - multiplier * iqr, upper=q3 + multiplier * iqr)

        out[col] = out.groupby(group_cols)[col].transform(_clip_group)

    return out

In [95]:
#  Clean the product-month modeling table while avoiding target leakage
def clean_modeling_dataset(datasets: dict[str, pd.DataFrame]) -> pd.DataFrame:
    df = datasets["ews_modeling_dataset_enriched"].copy()

    df = df.drop_duplicates(["product_id", "year_month", "region"]).copy()
    df = df.sort_values(["product_id", "region", "year_month"]).reset_index(drop=True)

    for col in NUMERIC_FEATURES + [TARGET_COL]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Business constraints.
    non_negative_cols = [
        "units_sold", "revenue", "unique_customers", "avg_discount_pct",
        "units_lag_1m", "units_lag_3m", "revenue_lag_1m",
        "rolling_units_mean_3m", "rolling_units_mean_6m", "rolling_units_std_3m",
        "volatility_3m", "stockout_flag", "backorder_units",
        "market_demand_index", "competitor_pressure_index", "campaign_flag",
        "website_visits", "demo_requests",
        "risk_factor_sales_drop", "risk_factor_customer_drop",
        "risk_factor_volatility", "risk_factor_under_trend",
    ]
    for col in non_negative_cols:
        df[col] = df[col].clip(lower=0)

    df["avg_discount_pct"] = df["avg_discount_pct"].clip(0, 0.60)
    df["competitor_pressure_index"] = df["competitor_pressure_index"].clip(0, 100)
    df["market_demand_index"] = df["market_demand_index"].clip(0, 200)
    df["stockout_flag"] = df["stockout_flag"].fillna(0).round().clip(0, 1)
    df["campaign_flag"] = df["campaign_flag"].fillna(0).round().clip(0, 1)

    # Cap heavy-tailed business measures, but keep real zeros.
    df = cap_outliers_iqr(
        df,
        columns=["units_sold", "revenue", "website_visits", "demo_requests"],
        group_cols=["product_category", "region"],
        multiplier=4.0,
    )

    # Add interpretable calendar features.
    df["year"] = df["year_month"].dt.year
    df["month"] = df["year_month"].dt.month
    df["quarter"] = df["year_month"].dt.quarter
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    # Add safe ratio features based only on current/historical values.
    df["revenue_per_unit"] = df["revenue"] / df["units_sold"].replace(0, np.nan)
    df["revenue_per_customer"] = df["revenue"] / df["unique_customers"].replace(0, np.nan)
    df["backorder_rate"] = df["backorder_units"] / (df["units_sold"] + df["backorder_units"]).replace(0, np.nan)
    df["marketing_intensity"] = df["website_visits"] / df["rolling_units_mean_3m"].replace(0, np.nan)

    engineered_numeric = [
        "year", "month", "quarter", "month_sin", "month_cos",
        "revenue_per_unit", "revenue_per_customer", "backorder_rate", "marketing_intensity",
    ]

    return df, NUMERIC_FEATURES + engineered_numeric

In [100]:
def run_preprocessing_pipeline():
    print("Loading raw synthetic EWS CSV files...")
    raw = load_csv_files()

    print("Standardizing dates and validating relational keys...")
    dated = standardize_dates(raw)
    key_checks = validate_keys(dated)

    print("Cleaning dimensions and transaction table...")
    cleaned = clean_dimension_tables(dated)
    cleaned = clean_transaction_table(cleaned)

    print("Cleaning final modeling dataset and creating safe engineered features...")
    modeling_clean, numeric_features = clean_modeling_dataset(cleaned)

    # Save a human-readable cleaned product-month dataset before one-hot/scaling.
    modeling_clean.to_csv(OUTPUT_DIR_PREPROCESSED / "final_modeling_dataset_cleaned.csv", index=False)

    print("Creating time-based train/validation/test splits...")
    train, valid, test = create_time_based_splits(modeling_clean)

    print("Fitting pandas preprocessor on train only and exporting ML-ready matrices...")
    split_summary = transform_and_save_ml_matrices(
        train=train,
        valid=valid,
        test=test,
        numeric_features=numeric_features,
        categorical_features=CATEGORICAL_FEATURES,
    )

    report = create_quality_report(
        raw_datasets=raw,
        cleaned_modeling=modeling_clean,
        key_checks=key_checks,
        split_summary=split_summary,
        numeric_features=numeric_features,
    )

    print("\nPreprocessing complete")
    print("-" * 80)
    print(f"Cleaned modeling dataset: {modeling_clean.shape}")
    print(f"ML feature count after preprocessing: {split_summary['feature_count']}")
    print(f"Train shape: {split_summary['train_shape']}")
    print(f"Validation shape: {split_summary['valid_shape']}")
    print(f"Test shape: {split_summary['test_shape']}")

    print("\nRelational key checks")
    print("-" * 80)
    for key, value in key_checks.items():
        print(f"{key}: {value}")

    print("\nTarget distribution")
    print("-" * 80)
    print(modeling_clean[TARGET_COL].value_counts(dropna=False).sort_index().to_string())

    print("\nTop missing rates after cleaning")
    print("-" * 80)
    print(pd.Series(report["missing_rate_top_15"]).to_string())

    print("\nOutput files saved to:", OUTPUT_DIR_PREPROCESSED.resolve())

    return {
        "raw": raw,
        "cleaned_modeling": modeling_clean,
        "train": train,
        "valid": valid,
        "test": test,
        "quality_report": report,
    }


In [109]:
artifacts = run_preprocessing_pipeline()

Loading raw synthetic EWS CSV files...
Standardizing dates and validating relational keys...
Cleaning dimensions and transaction table...
Cleaning final modeling dataset and creating safe engineered features...
Creating time-based train/validation/test splits...
Fitting pandas preprocessor on train only and exporting ML-ready matrices...

Preprocessing complete
--------------------------------------------------------------------------------
Cleaned modeling dataset: (78816, 89)
ML feature count after preprocessing: 60
Train shape: (50016, 61)
Validation shape: (14400, 61)
Test shape: (12800, 61)

Relational key checks
--------------------------------------------------------------------------------
duplicate_product_ids: 0
duplicate_customer_ids: 0
duplicate_transaction_ids: 0
sales_unknown_products: 0
sales_unknown_customers: 0
final_unknown_products: 0
final_duplicate_panel_rows: 0

Target distribution
--------------------------------------------------------------------------------
ne